# Notebook 3 : Deep Learning — MLP & CNN

Ce notebook implémente des modèles de **Deep Learning** pour la classification des avis Yelp.

**Modèles :**
- **MLP** (Multi-Layer Perceptron) — sur TF-IDF et Embeddings SBERT
- **CNN** (Convolutional Neural Network) — avec couche d’embedding entraînable

**Tâches :**
- Classification de polarité (3 classes : positive, negative, neutral)
- Classification de rating (5 classes : 1–5 étoiles)

**Comparaison avec les résultats ML classique (Notebook 04)**

## 1. Imports et Configuration

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# Embeddings pré-entraînés
from sentence_transformers import SentenceTransformer

# Tokenisation pour CNN
from collections import Counter

# Utilitaires projet
import sys
sys.path.append('../src')
from utils import preprocess_dataframe, load_json_lines

import os
import json
import time

# Configuration
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Device GPU/CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
if device.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# Style des plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Imports réussis")

Device : cpu
Imports réussis


## 2. Chargement et Preprocessing des Données

In [3]:
# Charger les données
df = load_json_lines('../data/raw/yelp_academic_reviews4students.jsonl')
print(f"Dataset original : {len(df)} reviews")

# Preprocessing
df = preprocess_dataframe(
    df,
    text_column='text',
    rating_column='stars',
    lowercase=True,
    remove_punctuation=False,
    remove_numbers=False,
    add_polarity=True,
    add_rating=True,
    min_words=10,
    truncate=False
)

print(f"Dataset après preprocessing : {len(df)} reviews")
print(f"\nPolarité :")
print(df['polarite'].value_counts())
print(f"\nRating :")
print(df['rating'].value_counts().sort_index())

  Chargé 100,000 lignes...
  Chargé 200,000 lignes...
  Chargé 300,000 lignes...
  Chargé 400,000 lignes...
  Chargé 500,000 lignes...
  Chargé 600,000 lignes...
  Chargé 700,000 lignes...
  Chargé 800,000 lignes...
  Chargé 900,000 lignes...
  Chargé 1,000,000 lignes...
Dataset original : 1000000 reviews
Dataset après preprocessing : 997911 reviews

Polarité :
polarite
positive    668925
negative    230466
neutral      98520
Name: count, dtype: int64

Rating :
rating
1    152916
2     77550
3     98520
4    207328
5    461597
Name: count, dtype: int64


## 3. Encodage des Labels et Train/Test Split

In [4]:
# Encodage des labels
le_polarity = LabelEncoder()
df['polarite_encoded'] = le_polarity.fit_transform(df['polarite'])
print(f"Classes polarité : {dict(zip(le_polarity.classes_, le_polarity.transform(le_polarity.classes_)))}")

le_rating = LabelEncoder()
df['rating_encoded'] = le_rating.fit_transform(df['rating'])
print(f"Classes rating   : {dict(zip(le_rating.classes_, le_rating.transform(le_rating.classes_)))}")

# Train/Test split
X_train_text, X_test_text, y_train_pol, y_test_pol = train_test_split(
    df['text_clean'],
    df['polarite_encoded'],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df['polarite_encoded']
)

_, _, y_train_rating, y_test_rating = train_test_split(
    df['text_clean'],
    df['rating_encoded'],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df['rating_encoded']
)

print(f"\nTrain : {len(X_train_text)} | Test : {len(X_test_text)}")

Classes polarité : {'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}
Classes rating   : {np.int64(1): np.int64(0), np.int64(2): np.int64(1), np.int64(3): np.int64(2), np.int64(4): np.int64(3), np.int64(5): np.int64(4)}

Train : 798328 | Test : 199583


## 4. Préparation des Représentations

On prépare trois types d'entrées pour nos modèles :
1. **TF-IDF** (vecteurs sparse → dense) pour le MLP
2. **Embeddings SBERT** (vecteurs denses 384-dim) pour le MLP
3. **Séquences de tokens** (indices de mots paddés) pour le CNN

### 4.1 TF-IDF

In [5]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2),
    stop_words='english',
    sublinear_tf=True
)

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)

print(f"TF-IDF train shape : {X_train_tfidf.shape} (sparse, {X_train_tfidf.nnz:,} non-zero)")
print(f"TF-IDF test shape  : {X_test_tfidf.shape} (sparse, {X_test_tfidf.nnz:,} non-zero)")

TF-IDF train shape : (798328, 5000) (sparse, 30,672,179 non-zero)
TF-IDF test shape  : (199583, 5000) (sparse, 7,653,531 non-zero)
Mémoire sparse     : ~371 Mo (vs ~16.0 Go dense)


### 4.2 Embeddings SBERT

In [ ]:
EMBEDDINGS_DIR = '../data/embeddings'
os.makedirs(EMBEDDINGS_DIR, exist_ok=True)

train_emb_path = os.path.join(EMBEDDINGS_DIR, 'sbert_train.npy')
test_emb_path = os.path.join(EMBEDDINGS_DIR, 'sbert_test.npy')

if os.path.exists(train_emb_path) and os.path.exists(test_emb_path):
    # Chargement depuis le cache (instantané)
    print("Chargement des embeddings depuis le cache...")
    X_train_emb = np.load(train_emb_path)
    X_test_emb = np.load(test_emb_path)
    print("Embeddings chargés depuis le cache")
else:
    # Calcul des embeddings (long ~30min CPU, ~5min GPU Colab)
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    print(f"Dimension des embeddings : {embedding_model.get_sentence_embedding_dimension()}")

    print("Encodage du train set...")
    X_train_emb = embedding_model.encode(
        X_train_text.tolist(),
        batch_size=256,
        show_progress_bar=True
    ).astype(np.float32)

    print("Encodage du test set...")
    X_test_emb = embedding_model.encode(
        X_test_text.tolist(),
        batch_size=256,
        show_progress_bar=True
    ).astype(np.float32)

    # Sauvegarder sur disque pour ne plus recalculer
    np.save(train_emb_path, X_train_emb)
    np.save(test_emb_path, X_test_emb)
    print(f"Embeddings sauvegardés dans {EMBEDDINGS_DIR}/")

print(f"\nEmbeddings train shape : {X_train_emb.shape}")
print(f"Embeddings test shape  : {X_test_emb.shape}")
print(f"Mémoire : ~{(X_train_emb.nbytes + X_test_emb.nbytes) / 1e6:.0f} Mo")

### 4.3 Tokenisation pour le CNN

Le CNN travaille sur des **séquences de mots** (indices), pas des vecteurs agrégés.  
On construit un vocabulaire et on convertit chaque review en séquence d’indices avec padding.

In [ ]:
MAX_VOCAB_SIZE = 25000
MAX_SEQ_LEN = 200

# Construire le vocabulaire sur le train set
word_counts = Counter()
for text in X_train_text:
    word_counts.update(text.split())

# Garder les mots les plus fréquents
vocab = {word: idx + 2 for idx, (word, _) in enumerate(word_counts.most_common(MAX_VOCAB_SIZE))}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1
VOCAB_SIZE = len(vocab)

print(f"Taille du vocabulaire : {VOCAB_SIZE}")
print(f"Longueur max des séquences : {MAX_SEQ_LEN}")

def text_to_sequence(text, vocab, max_len):
    """Convertit un texte en séquence d'indices avec padding."""
    tokens = text.split()
    indices = [vocab.get(w, 1) for w in tokens[:max_len]]  # 1 = <UNK>
    # Padding
    if len(indices) < max_len:
        indices += [0] * (max_len - len(indices))  # 0 = <PAD>
    return indices

X_train_seq = np.array([text_to_sequence(t, vocab, MAX_SEQ_LEN) for t in X_train_text])
X_test_seq = np.array([text_to_sequence(t, vocab, MAX_SEQ_LEN) for t in X_test_text])

print(f"Séquences train shape : {X_train_seq.shape}")
print(f"Séquences test shape  : {X_test_seq.shape}")
print(f"\nExemple de séquence (premiers 20 indices) : {X_train_seq[0][:20]}")

## 5. Datasets & DataLoaders PyTorch

In [ ]:
import scipy.sparse

class SparseDataset(Dataset):
    """Dataset pour matrices sparse scipy (TF-IDF). Convertit en dense ligne par ligne."""
    def __init__(self, sparse_matrix, labels):
        self.sparse_matrix = sparse_matrix.tocsr()
        self.labels = torch.tensor(labels.values, dtype=torch.long)
    
    def __len__(self):
        return self.sparse_matrix.shape[0]
    
    def __getitem__(self, idx):
        row = torch.tensor(self.sparse_matrix[idx].toarray().squeeze(0), dtype=torch.float32)
        return row, self.labels[idx]


class DenseDataset(Dataset):
    """Dataset pour arrays numpy denses (embeddings, séquences)."""
    def __init__(self, features, labels):
        if features.dtype == np.int64 or features.dtype == np.int32:
            self.features = torch.tensor(features, dtype=torch.long)
        else:
            self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels.values, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


BATCH_SIZE = 512

# --- Datasets pour POLARITÉ ---
# TF-IDF (sparse)
train_tfidf_pol = DataLoader(SparseDataset(X_train_tfidf, y_train_pol), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_tfidf_pol  = DataLoader(SparseDataset(X_test_tfidf, y_test_pol), batch_size=BATCH_SIZE, num_workers=0)

# Embeddings (dense)
train_emb_pol = DataLoader(DenseDataset(X_train_emb, y_train_pol), batch_size=BATCH_SIZE, shuffle=True)
test_emb_pol  = DataLoader(DenseDataset(X_test_emb, y_test_pol), batch_size=BATCH_SIZE)

# Séquences (pour CNN)
train_seq_pol = DataLoader(DenseDataset(X_train_seq, y_train_pol), batch_size=BATCH_SIZE, shuffle=True)
test_seq_pol  = DataLoader(DenseDataset(X_test_seq, y_test_pol), batch_size=BATCH_SIZE)

# --- Datasets pour RATING ---
train_tfidf_rat = DataLoader(SparseDataset(X_train_tfidf, y_train_rating), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_tfidf_rat  = DataLoader(SparseDataset(X_test_tfidf, y_test_rating), batch_size=BATCH_SIZE, num_workers=0)

train_emb_rat = DataLoader(DenseDataset(X_train_emb, y_train_rating), batch_size=BATCH_SIZE, shuffle=True)
test_emb_rat  = DataLoader(DenseDataset(X_test_emb, y_test_rating), batch_size=BATCH_SIZE)

train_seq_rat = DataLoader(DenseDataset(X_train_seq, y_train_rating), batch_size=BATCH_SIZE, shuffle=True)
test_seq_rat  = DataLoader(DenseDataset(X_test_seq, y_test_rating), batch_size=BATCH_SIZE)

print(f"DataLoaders créés (batch_size={BATCH_SIZE})")
print(f"  TF-IDF   : sparse -> dense à la volée (par batch)")
print(f"  Polarité : {len(le_polarity.classes_)} classes")
print(f"  Rating   : {len(le_rating.classes_)} classes")

## 6. Fonctions d’Entraînement et d’Évaluation

In [ ]:
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3, class_weights=None):
    """
    Boucle d'entraînement générique pour un modèle PyTorch.
    
    Returns:
        dict: historique (train_loss, val_loss, val_acc par époque)
    """
    model.to(device)
    
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    else:
        criterion = nn.CrossEntropyLoss()
    
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.5)
    
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0
    best_state = None
    
    for epoch in range(epochs):
        # --- Train ---
        model.train()
        total_loss = 0
        n_batches = 0
        
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            n_batches += 1
        
        avg_train_loss = total_loss / n_batches
        
        # --- Validation ---
        model.eval()
        val_loss = 0
        correct = 0
        total = 0
        n_val = 0
        
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                val_loss += criterion(outputs, y_batch).item()
                _, predicted = torch.max(outputs, 1)
                correct += (predicted == y_batch).sum().item()
                total += y_batch.size(0)
                n_val += 1
        
        avg_val_loss = val_loss / n_val
        val_acc = correct / total
        
        scheduler.step(avg_val_loss)
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        
        # Sauvegarder le meilleur modèle
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = model.state_dict().copy()
        
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"Val Acc: {val_acc:.4f} | "
              f"LR: {current_lr:.1e}")
    
    # Restaurer le meilleur modèle
    if best_state is not None:
        model.load_state_dict(best_state)
    
    print(f"\nMeilleure Val Acc : {best_val_acc:.4f}")
    return history


def evaluate_dl_model(model, test_loader, label_encoder, model_name):
    """
    Évalue un modèle PyTorch et affiche les métriques.
    
    Returns:
        dict: métriques (accuracy, f1_macro, f1_weighted)
    """
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            outputs = model(X_batch)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    acc = accuracy_score(all_labels, all_preds)
    f1_mac = f1_score(all_labels, all_preds, average='macro')
    f1_w = f1_score(all_labels, all_preds, average='weighted')
    
    print(f"\n{'='*60}")
    print(f" {model_name}")
    print(f"{'='*60}")
    print(f"Accuracy:     {acc:.4f}")
    print(f"F1-macro:     {f1_mac:.4f}")
    print(f"F1-weighted:  {f1_w:.4f}")
    
    target_names = label_encoder.classes_.astype(str)
    print(f"\n{classification_report(all_labels, all_preds, target_names=target_names)}")
    
    # Matrice de confusion
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=target_names, yticklabels=target_names)
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    return {'model': model_name, 'accuracy': acc, 'f1_macro': f1_mac, 'f1_weighted': f1_w}


def plot_training_history(history, title):
    """Affiche les courbes de loss et accuracy."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
    axes[0].set_title(f'{title} - Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(history['val_acc'], label='Val Accuracy', marker='o', color='green')
    axes[1].set_title(f'{title} - Validation Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


# Calcul des poids de classes pour gérer le déséquilibre
def compute_class_weights(labels):
    """Calcule les poids inversés pour chaque classe."""
    counts = np.bincount(labels.values)
    weights = 1.0 / counts
    weights = weights / weights.sum() * len(counts)
    return torch.tensor(weights, dtype=torch.float32)

weights_pol = compute_class_weights(y_train_pol)
weights_rat = compute_class_weights(y_train_rating)
print(f"Poids polarité : {weights_pol}")
print(f"Poids rating   : {weights_rat}")

# Stocker les résultats
results_pol = []
results_rat = []

print("\nFonctions d'évaluation définies")

---
## 7. MLP (Multi-Layer Perceptron)

Réseau de neurones entièrement connecté (feedforward).  
On l’entraîne sur deux types d’entrées : **TF-IDF** et **Embeddings SBERT**.

In [ ]:
class MLP(nn.Module):
    """
    Multi-Layer Perceptron pour la classification de texte.
    
    Architecture :
        Input -> FC(256) -> ReLU -> Dropout -> FC(128) -> ReLU -> Dropout -> FC(n_classes)
    """
    def __init__(self, input_dim, n_classes, hidden_dims=(256, 128), dropout=0.3):
        super(MLP, self).__init__()
        
        layers = []
        prev_dim = input_dim
        
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = h_dim
        
        layers.append(nn.Linear(prev_dim, n_classes))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

print("Architecture MLP :")
print(MLP(5000, 3))

### 7.1 MLP + TF-IDF → Polarité

In [ ]:
EPOCHS_MLP = 10

mlp_tfidf_pol = MLP(input_dim=5000, n_classes=3, hidden_dims=(256, 128), dropout=0.3)
print(f"Paramètres : {sum(p.numel() for p in mlp_tfidf_pol.parameters()):,}")

history_mlp_tfidf_pol = train_model(
    mlp_tfidf_pol, train_tfidf_pol, test_tfidf_pol,
    epochs=EPOCHS_MLP, lr=1e-3, class_weights=weights_pol
)

plot_training_history(history_mlp_tfidf_pol, "MLP + TF-IDF (Polarité)")
results_pol.append(evaluate_dl_model(mlp_tfidf_pol, test_tfidf_pol, le_polarity, "MLP + TF-IDF"))

### 7.2 MLP + Embeddings SBERT → Polarité

In [ ]:
mlp_emb_pol = MLP(input_dim=384, n_classes=3, hidden_dims=(256, 128), dropout=0.3)
print(f"Paramètres : {sum(p.numel() for p in mlp_emb_pol.parameters()):,}")

history_mlp_emb_pol = train_model(
    mlp_emb_pol, train_emb_pol, test_emb_pol,
    epochs=EPOCHS_MLP, lr=1e-3, class_weights=weights_pol
)

plot_training_history(history_mlp_emb_pol, "MLP + Embeddings (Polarité)")
results_pol.append(evaluate_dl_model(mlp_emb_pol, test_emb_pol, le_polarity, "MLP + Embeddings"))

### 7.3 MLP + TF-IDF → Rating

In [ ]:
mlp_tfidf_rat = MLP(input_dim=5000, n_classes=5, hidden_dims=(256, 128), dropout=0.3)
print(f"Paramètres : {sum(p.numel() for p in mlp_tfidf_rat.parameters()):,}")

history_mlp_tfidf_rat = train_model(
    mlp_tfidf_rat, train_tfidf_rat, test_tfidf_rat,
    epochs=EPOCHS_MLP, lr=1e-3, class_weights=weights_rat
)

plot_training_history(history_mlp_tfidf_rat, "MLP + TF-IDF (Rating)")
results_rat.append(evaluate_dl_model(mlp_tfidf_rat, test_tfidf_rat, le_rating, "MLP + TF-IDF"))

### 7.4 MLP + Embeddings SBERT → Rating

In [ ]:
mlp_emb_rat = MLP(input_dim=384, n_classes=5, hidden_dims=(256, 128), dropout=0.3)
print(f"Paramètres : {sum(p.numel() for p in mlp_emb_rat.parameters()):,}")

history_mlp_emb_rat = train_model(
    mlp_emb_rat, train_emb_rat, test_emb_rat,
    epochs=EPOCHS_MLP, lr=1e-3, class_weights=weights_rat
)

plot_training_history(history_mlp_emb_rat, "MLP + Embeddings (Rating)")
results_rat.append(evaluate_dl_model(mlp_emb_rat, test_emb_rat, le_rating, "MLP + Embeddings"))

---
## 8. CNN (Convolutional Neural Network)

Architecture inspirée de **"Convolutional Neural Networks for Sentence Classification" (Yoon Kim, 2014)** :  
- Couche d’embedding entraînable
- Convolutions 1D multi-filtres (tailles 3, 4, 5) pour capturer les n-grammes
- Max-pooling global
- Couche fully-connected de sortie

In [ ]:
class TextCNN(nn.Module):
    """
    CNN pour la classification de texte (Yoon Kim, 2014).
    
    Architecture :
        Embedding -> Conv1D (multi-filtres) -> MaxPool -> FC -> Output
    """
    def __init__(self, vocab_size, embed_dim, n_classes, 
                 n_filters=128, filter_sizes=(3, 4, 5), dropout=0.4):
        super(TextCNN, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # Convolutions avec différentes tailles de filtre
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=n_filters, kernel_size=fs)
            for fs in filter_sizes
        ])
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(n_filters * len(filter_sizes), n_classes)
    
    def forward(self, x):
        # x shape: (batch, seq_len)
        embedded = self.embedding(x)        # (batch, seq_len, embed_dim)
        embedded = embedded.permute(0, 2, 1) # (batch, embed_dim, seq_len) pour Conv1d
        
        # Appliquer chaque convolution + ReLU + MaxPool
        conv_outputs = []
        for conv in self.convs:
            c = torch.relu(conv(embedded))   # (batch, n_filters, seq_len - fs + 1)
            c = c.max(dim=2).values          # Global max pooling -> (batch, n_filters)
            conv_outputs.append(c)
        
        # Concaténer les sorties des différents filtres
        out = torch.cat(conv_outputs, dim=1)  # (batch, n_filters * len(filter_sizes))
        out = self.dropout(out)
        out = self.fc(out)
        return out

EMBED_DIM = 128
N_FILTERS = 128
FILTER_SIZES = (3, 4, 5)

print("Architecture TextCNN :")
print(TextCNN(VOCAB_SIZE, EMBED_DIM, 3, N_FILTERS, FILTER_SIZES))
print(f"\nParamètres : {sum(p.numel() for p in TextCNN(VOCAB_SIZE, EMBED_DIM, 3, N_FILTERS, FILTER_SIZES).parameters()):,}")

### 8.1 CNN → Polarité

In [ ]:
EPOCHS_CNN = 8

cnn_pol = TextCNN(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    n_classes=3,
    n_filters=N_FILTERS,
    filter_sizes=FILTER_SIZES,
    dropout=0.4
)

history_cnn_pol = train_model(
    cnn_pol, train_seq_pol, test_seq_pol,
    epochs=EPOCHS_CNN, lr=1e-3, class_weights=weights_pol
)

plot_training_history(history_cnn_pol, "CNN (Polarité)")
results_pol.append(evaluate_dl_model(cnn_pol, test_seq_pol, le_polarity, "CNN"))

### 8.2 CNN → Rating

In [ ]:
cnn_rat = TextCNN(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    n_classes=5,
    n_filters=N_FILTERS,
    filter_sizes=FILTER_SIZES,
    dropout=0.4
)

history_cnn_rat = train_model(
    cnn_rat, train_seq_rat, test_seq_rat,
    epochs=EPOCHS_CNN, lr=1e-3, class_weights=weights_rat
)

plot_training_history(history_cnn_rat, "CNN (Rating)")
results_rat.append(evaluate_dl_model(cnn_rat, test_seq_rat, le_rating, "CNN"))

---
## 9. Comparaison des Résultats Deep Learning

In [ ]:
# --- Résultats Polarité ---
df_pol = pd.DataFrame(results_pol).sort_values('accuracy', ascending=False)
print("\n" + "="*60)
print(" RÉSULTATS POLARITÉ (3 classes)")
print("="*60)
print(df_pol.to_string(index=False))

best_pol = df_pol.iloc[0]
print(f"\nMeilleur modèle : {best_pol['model']}")
print(f"  Accuracy: {best_pol['accuracy']:.4f} | F1-macro: {best_pol['f1_macro']:.4f}")

# --- Résultats Rating ---
df_rat = pd.DataFrame(results_rat).sort_values('accuracy', ascending=False)
print("\n" + "="*60)
print(" RÉSULTATS RATING (5 classes)")
print("="*60)
print(df_rat.to_string(index=False))

best_rat = df_rat.iloc[0]
print(f"\nMeilleur modèle : {best_rat['model']}")
print(f"  Accuracy: {best_rat['accuracy']:.4f} | F1-macro: {best_rat['f1_macro']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Polarité
df_pol_sorted = df_pol.sort_values('f1_macro')
colors_pol = ['#ff9999' if 'MLP' in m else '#66b3ff' for m in df_pol_sorted['model']]
axes[0].barh(df_pol_sorted['model'], df_pol_sorted['f1_macro'], color=colors_pol)
axes[0].set_title('Polarité (3 classes) - F1-Macro', fontsize=14, fontweight='bold')
axes[0].set_xlabel('F1-Macro')
axes[0].set_xlim([0, 1])
for i, v in enumerate(df_pol_sorted['f1_macro']):
    axes[0].text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=10)
axes[0].grid(axis='x', alpha=0.3)

# Rating
df_rat_sorted = df_rat.sort_values('f1_macro')
colors_rat = ['#ff9999' if 'MLP' in m else '#66b3ff' for m in df_rat_sorted['model']]
axes[1].barh(df_rat_sorted['model'], df_rat_sorted['f1_macro'], color=colors_rat)
axes[1].set_title('Rating (5 classes) - F1-Macro', fontsize=14, fontweight='bold')
axes[1].set_xlabel('F1-Macro')
axes[1].set_xlim([0, 1])
for i, v in enumerate(df_rat_sorted['f1_macro']):
    axes[1].text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=10)
axes[1].grid(axis='x', alpha=0.3)

# Légende
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#ff9999', label='MLP'), Patch(facecolor='#66b3ff', label='CNN')]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, fontsize=12,
           bbox_to_anchor=(0.5, 1.02))

plt.tight_layout()
os.makedirs('../figures', exist_ok=True)
plt.savefig('../figures/deep_learning_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Graphique sauvegardé : ../figures/deep_learning_comparison.png")

## 10. Comparaison avec le ML Classique (Notebook 04)

On charge les résultats du notebook 04 pour comparer ML classique vs Deep Learning.

In [ ]:
# Charger les résultats ML classique s'ils existent
ml_results_path = '../results/ml_classique_results.csv'

if os.path.exists(ml_results_path):
    ml_results = pd.read_csv(ml_results_path)
    ml_results['type'] = 'ML Classique'
    
    dl_pol_df = df_pol.copy()
    dl_pol_df['type'] = 'Deep Learning'
    
    all_results = pd.concat([ml_results, dl_pol_df], ignore_index=True)
    all_results = all_results.sort_values('f1_macro', ascending=False)
    
    print("\n" + "="*70)
    print(" COMPARAISON COMPLÈTE : ML Classique vs Deep Learning (Polarité)")
    print("="*70)
    print(all_results[['model', 'type', 'accuracy', 'f1_macro', 'f1_weighted']].to_string(index=False))
    
    # Visualisation
    fig, ax = plt.subplots(figsize=(14, 7))
    colors = ['#ff9999' if t == 'ML Classique' else '#66b3ff' for t in all_results['type']]
    
    all_sorted = all_results.sort_values('f1_macro')
    colors_sorted = ['#ff9999' if t == 'ML Classique' else '#66b3ff' for t in all_sorted['type']]
    
    ax.barh(all_sorted['model'], all_sorted['f1_macro'], color=colors_sorted)
    ax.set_title('ML Classique vs Deep Learning - F1-Macro (Polarité)', fontsize=14, fontweight='bold')
    ax.set_xlabel('F1-Macro', fontsize=12)
    ax.set_xlim([0, 1])
    
    for i, (v, t) in enumerate(zip(all_sorted['f1_macro'], all_sorted['type'])):
        ax.text(v + 0.005, i, f'{v:.4f}', va='center', fontsize=9)
    
    legend_elements = [
        Patch(facecolor='#ff9999', label='ML Classique'),
        Patch(facecolor='#66b3ff', label='Deep Learning')
    ]
    ax.legend(handles=legend_elements, fontsize=11)
    ax.grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../figures/ml_vs_dl_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("Graphique sauvegardé : ../figures/ml_vs_dl_comparison.png")
else:
    print("Résultats ML classique non trouvés. Exécutez d'abord le Notebook 04.")
    print("\nRésultats Deep Learning uniquement :")
    print(df_pol.to_string(index=False))

## 11. Sauvegarde des Modèles et Résultats

In [ ]:
os.makedirs('../models', exist_ok=True)
os.makedirs('../results', exist_ok=True)

# Sauvegarder les modèles PyTorch
torch.save(mlp_tfidf_pol.state_dict(), '../models/mlp_tfidf_pol.pt')
torch.save(mlp_emb_pol.state_dict(), '../models/mlp_emb_pol.pt')
torch.save(cnn_pol.state_dict(), '../models/cnn_pol.pt')
torch.save(mlp_tfidf_rat.state_dict(), '../models/mlp_tfidf_rat.pt')
torch.save(mlp_emb_rat.state_dict(), '../models/mlp_emb_rat.pt')
torch.save(cnn_rat.state_dict(), '../models/cnn_rat.pt')
print("Modèles PyTorch sauvegardés dans ../models/")

# Sauvegarder les résultats
df_pol.to_csv('../results/deep_learning_polarite_results.csv', index=False)
df_rat.to_csv('../results/deep_learning_rating_results.csv', index=False)
print("Résultats sauvegardés dans ../results/")

# Sauvegarder le vocabulaire (nécessaire pour réutiliser le CNN)
import pickle
with open('../models/cnn_vocab.pkl', 'wb') as f:
    pickle.dump(vocab, f)
print("Vocabulaire CNN sauvegardé : ../models/cnn_vocab.pkl")

# Métadonnées
metadata = {
    'best_polarity_model': best_pol['model'],
    'best_polarity_accuracy': float(best_pol['accuracy']),
    'best_polarity_f1_macro': float(best_pol['f1_macro']),
    'best_rating_model': best_rat['model'],
    'best_rating_accuracy': float(best_rat['accuracy']),
    'best_rating_f1_macro': float(best_rat['f1_macro']),
    'device': str(device),
    'train_size': len(X_train_text),
    'test_size': len(X_test_text),
    'mlp_epochs': EPOCHS_MLP,
    'cnn_epochs': EPOCHS_CNN,
    'batch_size': BATCH_SIZE,
    'cnn_vocab_size': VOCAB_SIZE,
    'cnn_embed_dim': EMBED_DIM,
    'cnn_max_seq_len': MAX_SEQ_LEN
}

with open('../results/deep_learning_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Métadonnées sauvegardées : ../results/deep_learning_metadata.json")

## 12. Analyse des Courbes d’Entraînement

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))

histories = [
    (history_mlp_tfidf_pol, "MLP+TF-IDF (Pol)"),
    (history_mlp_emb_pol, "MLP+Emb (Pol)"),
    (history_cnn_pol, "CNN (Pol)"),
    (history_mlp_tfidf_rat, "MLP+TF-IDF (Rat)"),
    (history_mlp_emb_rat, "MLP+Emb (Rat)"),
    (history_cnn_rat, "CNN (Rat)"),
]

for idx, (hist, title) in enumerate(histories):
    row, col = idx // 3, idx % 3
    ax = axes[row][col]
    
    ax.plot(hist['train_loss'], label='Train Loss', marker='o', markersize=4)
    ax.plot(hist['val_loss'], label='Val Loss', marker='s', markersize=4)
    
    ax2 = ax.twinx()
    ax2.plot(hist['val_acc'], label='Val Acc', color='green', marker='^', markersize=4, linestyle='--')
    ax2.set_ylabel('Accuracy', color='green')
    ax2.tick_params(axis='y', labelcolor='green')
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(loc='upper left', fontsize=8)
    ax2.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Courbes d\'entraînement - Tous les modèles', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/training_curves_all.png', dpi=300, bbox_inches='tight')
plt.show()
print("Graphique sauvegardé : ../figures/training_curves_all.png")

## 13. Résumé et Conclusions

### Résultats clés

| Modèle | Représentation | Avantages | Limites |
|--------|---------------|-----------|----------|
| **MLP + TF-IDF** | Vecteurs TF-IDF (5000-dim) | Simple, rapide à entraîner | Perd l’ordre des mots |
| **MLP + Embeddings** | SBERT (384-dim) | Représentation sémantique riche | Dépend du modèle pré-entraîné |
| **CNN** | Embedding entraînable + Conv1D | Capture les n-grammes locaux | Plus de paramètres, plus lent |

### Points à retenir

- Le **MLP** est un bon point de départ en Deep Learning : il montre déjà des gains par rapport au ML classique grâce à sa capacité à apprendre des non-linéarités.
- Le **CNN** capture mieux le contexte local (n-grammes) via les convolutions multi-filtres.
- La tâche de **rating (5 classes)** reste plus difficile que la **polarité (3 classes)**, les classes voisines (3 vs 4, 2 vs 3) étant souvent confondues.
- Les **poids de classes** aident à gérer le déséquilibre du dataset Yelp.

### Prochaines étapes

- **Notebook 06** : Modèles Transformer (BERT fine-tuné) pour pousser les performances encore plus loin.
- **Notebook 07** : IA générative (zero-shot, few-shot, extraction d’aspects avec LLM).